# 인바운드 인증에 Gateway IAM 역할을 사용하여 AWS Lambda를 MCP로 변환하기
## Bedrock AgentCore Gateway를 사용하여 AWS Lambda 함수를 안전한 MCP 도구로 변환하기

## 개요
Bedrock AgentCore Gateway는 고객이 인프라나 호스팅을 관리할 필요 없이 기존 AWS Lambda 함수를 완전 관리형 MCP 서버로 전환할 수 있는 방법을 제공합니다. Gateway는 이러한 모든 도구에 일관된 Model Context Protocol(MCP) 인터페이스를 제공합니다. Gateway는 수신 요청과 대상 리소스로의 아웃바운드 연결 모두에 안전한 액세스 제어를 보장하기 위해 이중 인증 모델을 사용합니다. 이 프레임워크는 Gateway 대상에 액세스하려는 사용자를 검증하고 권한을 부여하는 인바운드 인증과, 인증된 사용자를 대신하여 Gateway가 백엔드 리소스에 안전하게 연결할 수 있도록 하는 아웃바운드 인증이라는 두 가지 핵심 구성 요소로 이루어집니다. Gateway는 아웃바운드 권한 부여를 위해 IAM 역할을 사용하여 AWS Lambda 함수 호출을 승인합니다.

이 예제에서는 IAM 역할을 사용한 인바운드 및 아웃바운드 권한 부여를 모두 살펴봅니다.

![작동 방식](images/lambda-gw-iam-inbound.png)

### 튜토리얼 세부 정보


| 정보                 | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형        | 대화형                                                     |
| AgentCore 구성 요소  | AgentCore Gateway                                         |
| 에이전틱 프레임워크  | Strands Agents                                            |
| Gateway 대상 유형    | AWS Lambda                                                |
| 인바운드 인증        | AAWS IAM                                                  |
| 아웃바운드 인증      | AWS IAM                                                   |
| LLM 모델             | Anthropic Claude Haiku 4.5, Amazon Nova Pro              |
| 튜토리얼 구성 요소   | AgentCore Gateway 생성 및 AgentCore Gateway 호출          |
| 튜토리얼 분야        | 여러 분야                                                 |
| 예제 난이도          | 쉬움                                                      |
| 사용 SDK             | boto3                                                     |

튜토리얼의 첫 번째 부분에서는 Lambda용 AmazonCore Gateway 대상을 생성합니다.

### 튜토리얼 아키텍처
이 튜토리얼에서는 AWS Lambda 함수에 정의된 작업을 MCP 도구로 변환하고 Bedrock AgentCore Gateway에서 호스팅합니다. AWS Sigv4 헤더의 AWS IAM 자격 증명을 사용하는 인그레스 인증을 살펴봅니다.
시연을 위해 Amazon Bedrock 모델을 사용하는 Strands Agent를 사용합니다.
이 예제에서는 get_order와 update_order라는 두 가지 도구를 사용하는 매우 간단한 에이전트를 사용합니다.

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
* Jupyter notebook(Python 커널)
* uv
* AWS 자격 증명
* AWS 콘솔을 통한 Nova Pro 액세스 권한
* Amazon Bedrock AgentCore SDK
* Strand Agents

## 수신 AgentCore Gateway 요청에 대한 인증 구성
AgentCore Gateway는 인바운드 및 아웃바운드 인증을 통해 안전한 연결을 제공합니다. 인바운드 인증의 경우 이제 AgentCore Gateway를 호출할 때 OAuth뿐 아니라 AWS IAM 자격 증명/아이덴티티도 지원합니다. 도구가 외부 리소스에 액세스해야 하는 경우 AgentCore Gateway는 API Key, IAM 또는 OAuth Token을 통한 아웃바운드 인증을 사용하여 외부 리소스에 대한 액세스를 허용하거나 거부할 수 있습니다.

인바운드 권한 부여 흐름 중에 에이전트 또는 MCP 클라이언트는 인증에 사용되는 AWS Signature V4 서명 요청을 보내며, IAM 권한을 기준으로 AgentCore Gateway 액세스 권한을 부여받습니다. 그러면 AgentCore Gateway가 AWS IAM 자격 증명/아이덴티티를 검증하고 인바운드 권한 부여를 수행합니다.

AgentCore Gateway에서 실행되는 도구가 외부 리소스에 액세스해야 하는 경우 IAM 역할은 Gateway 대상을 위한 다운스트림 리소스의 자격 증명을 가져옵니다. AgentCore Gateway는 호출자가 다운스트림 API에 액세스할 수 있도록 권한 부여 자격 증명을 전달합니다.

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Amazon SageMaker notebook을 사용하지 않는 경우 AWS 자격 증명 설정
import os

# os.environ['AWS_ACCESS_KEY_ID']=''
# os.environ['AWS_SECRET_ACCESS_KEY']=''
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"  # AWS 리전 설정

In [ ]:
import os
import sys

# 현재 스크립트의 디렉터리 가져오기
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우 대체 경로 사용(예: Jupyter)

# utils.py가 있는 디렉터리로 이동(한 단계 위)
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

In [ ]:
# 이제 utils를 가져올 수 있습니다.
import utils

#### MCP 도구로 변환할 샘플 AWS Lambda 함수 생성
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")
if lambda_resp is not None:
    if lambda_resp["exit_code"] == 0:
        print("Lambda function created with ARN: ", lambda_resp["lambda_function_arn"])
    else:
        print(
            "Lambda function creation failed with message: ",
            lambda_resp["lambda_function_arn"],
        )

In [ ]:
#### Gateway가 수임할 IAM 역할 생성
import utils

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

# 인바운드 권한 부여를 위한 Amazon IAM Authorizer로 Gateway 생성

In [ ]:
import time
import boto3

# Amazon IAM으로 CreateGateway를 호출합니다.
gateway_client = boto3.client("bedrock-agentcore-control", region_name=os.environ["AWS_DEFAULT_REGION"])

create_response = gateway_client.create_gateway(
    name="TestGWforLambdaIAM",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway 생성/목록 조회/조회/삭제 권한이 있어야 합니다.
    protocolType="MCP",
    authorizerType="AWS_IAM",
    description="AgentCore Gateway with AWS Lambda target type using Amazon IAM for ingress auth",
)
print(create_response)
# GatewayTarget 생성에 사용할 GatewayID 가져오기
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)
time.sleep(10)

# AWS Lambda 대상을 생성하고 MCP 도구로 변환

In [ ]:
# 아래 AWS Lambda 함수 ARN을 교체합니다.
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp["lambda_function_arn"],  # 사용 중인 AWS Lambda 함수 ARN으로 교체합니다.
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"orderId": {"type": "string"}},
                            "required": ["orderId"],
                        },
                    },
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"orderId": {"type": "string"}},
                            "required": ["orderId"],
                        },
                    },
                ]
            },
        }
    }
}

credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]
targetname = "LambdaUsingSDK"
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description="Lambda Target using SDK",
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config,
)

# Gateway 호출을 위한 AWS IAM 역할 생성

아래 함수는 AWS Bedrock AgentCore가 지정된 Gateway를 호출할 수 있도록 IAM 역할을 생성하거나 업데이트합니다. 지정된 Gateway ID에 대한 bedrock-agentcore:InvokeGateway 권한을 부여하는 인라인 정책을 생성하여 연결합니다. 또한 Bedrock AgentCore 서비스와 호출하는 IAM 엔터티(current_arn)가 모두 역할을 수임할 수 있도록 신뢰 정책을 구성합니다.

In [ ]:
#### Gateway 호출을 위한 IAM 역할 생성
current_role_arn = utils.get_current_role_arn()
print("Current role ARN: ", current_role_arn)

agentcore_gateway_iam_invoke_role = utils.create_gateway_invoke_tool_role(
    "gateway-invoke-role", gatewayID, current_role_arn
)
print(
    "Role to invoke Agentcore gateway ARN: ",
    agentcore_gateway_iam_invoke_role["Role"]["Arn"],
)

# Bedrock AgentCore Gateway를 사용하여 AWS Lambda의 MCP 도구를 호출하는 Strands 에이전트

#### MCP Client SDK의 AWS IAM 인증 지원

이제 AgentCore Gateway로 보내는 인바운드 요청에서 AWS IAM 인증을 지원하지만, 현재 오픈 소스 MCP Client SDK는 특히 스트리밍 가능한 HTTP 연결에 대한 SigV4 인증 지원이 제한적이라는 점에 유의해야 합니다. 하지만 AWS는 스트리밍 HTTP 연결의 SigV4 인증에 필요한 핵심 확장 기능을 포함하는 "Run Model Context Protocol (MCP) servers with AWS Lambda" 프로젝트를 통해 해결 방법을 제공합니다.

[AWS Labs GitHub 리포지토리](https://github.com/awslabs/run-model-context-protocol-servers-with-aws-lambda/tree/main)에서 제공하는 이 구현은 스트리밍 연결의 인증 공백을 해소하며 Strands 또는 LangChain과 같은 널리 사용되는 에이전틱 프레임워크와 원활하게 통합할 수 있습니다. StreamableHTTPTransportWithSigV4 클래스는 표준 MCP 전송 계층을 확장하여 스트리밍 기능을 유지하면서 AWS SigV4 서명을 처리하므로 AgentCore Gateway의 새로운 IAM 인증 기능과 호환됩니다.

In [ ]:
!pip3 install --upgrade strands-agents strands-agents-tools
from strands.models import BedrockModel

## ~/.aws/credentials에 구성된 IAM 자격 증명에는 Bedrock 모델 액세스 권한이 있어야 합니다.
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
from strands import Agent
import logging
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from botocore.credentials import Credentials
from streamable_http_sigv4 import (
    streamablehttp_client_with_sigv4,
)

SERVICE = "bedrock-agentcore"

# 루트 strands 로거를 구성합니다. 문제를 디버깅하는 경우 DEBUG로 변경합니다.
logging.getLogger("strands").setLevel(logging.INFO)

# 로그를 확인할 수 있도록 핸들러 추가
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])


def create_streamable_http_transport(mcp_url: str, access_token: str):
    return streamablehttp_client(mcp_url, headers={"Authorization": f"Bearer {access_token}"})


def create_streamable_http_transport_sigv4(
    mcp_url: str,
    key: str,
    secret: str,
    sessionToken: str,
    serviceName: str,
    awsRegion: str,
):
    iamcredentials = Credentials(access_key=key, secret_key=secret, token=sessionToken)
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=iamcredentials,
        service=serviceName,
        region=awsRegion,
    )


def get_full_tools_list(client):
    more_tools = True
    tools = []
    pagination_token = None
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)
        tools.extend(tmp_tools)
        if tmp_tools.pagination_token is None:
            more_tools = False
        else:
            more_tools = True
            pagination_token = tmp_tools.pagination_token
    return tools


def call_tool_sync(client, tool_id, tool_name, parameters=None):
    # 도구 호출(페이지네이션 인수는 지원하지 않음)
    response = client.call_tool_sync(tool_use_id=tool_id, name=tool_name, arguments=parameters)

    # 출력 콘텐츠 추출
    if hasattr(response, "results") and response.results:
        return response.results
    elif hasattr(response, "output") and response.output:
        return response.output
    elif hasattr(response, "content"):
        return response.content
    else:
        return response  # 대체 반환값


def run_agent(
    mcp_url: str,
    key: str,
    secret: str,
    sessionToken: str,
    serviceName: str,
    awsRegion: str,
):
    mcp_client = MCPClient(
        lambda: create_streamable_http_transport_sigv4(mcp_url, key, secret, sessionToken, serviceName, awsRegion)
    )

    with mcp_client:
        tools = get_full_tools_list(mcp_client)
        print(f"Found the following tools: {[tool.tool_name for tool in tools]}")
        print(f"Tool name: {tools[0].tool_name}")

        agent = Agent(model=yourmodel, tools=tools)  ## 원하는 모델로 교체할 수 있습니다.
        print(f"Tools loaded in the agent are {agent.tool_names}")
        agent("Check the order status for order id 123 and show me the exact response from the tool")
        # 도구를 사용하여 MCP 호출
        tool = tools[0].tool_name
        tool_id = "get-order-id-123-call-1"
        result = call_tool_sync(mcp_client, tool_id, tool_name=tool, parameters={"orderId": "123"})

        print(f"Tool Call result: {result['content'][0]['text']}")

IAM Gateway 호출 역할을 수임하고 에이전트 실행

In [ ]:
sts_client = boto3.client("sts")
response = sts_client.assume_role(
    RoleArn=agentcore_gateway_iam_invoke_role["Role"]["Arn"],
    RoleSessionName="invoke_mcp_session",
    DurationSeconds=3600,  # 1시간, 일부 역할의 경우 최대 12시간까지 가능
)

creds = response["Credentials"]

access = creds["AccessKeyId"]
secret = creds["SecretAccessKey"]
token = creds["SessionToken"]

# 새 gateway-invoke-role의 자격 증명으로 에이전트 실행
time.sleep(10)
run_agent(gatewayURL, access, secret, token, SERVICE, os.environ["AWS_DEFAULT_REGION"])

**문제: 아래 셀을 실행할 때 다음 오류가 발생하면 pydantic과 pydantic-core 버전이 호환되지 않는다는 의미입니다.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**해결 방법**

서로 호환되는 pydantic==2.7.2와 pydantic-core 2.27.2가 설치되어 있는지 확인해야 합니다. 완료한 후 커널을 다시 시작합니다.

# 정리

IAM 역할, IAM 정책, 자격 증명 공급자, AWS Lambda 함수와 같은 추가 리소스도 생성되며 정리 과정에서 수동으로 삭제해야 할 수 있습니다. 이는 실행한 예제에 따라 달라집니다.

## Gateway 삭제(선택 사항)

In [ ]:
import utils

utils.delete_gateway(gateway_client, gatewayID)